# CVE Spectral Drift — Yearly Monitoring

This notebook turns the two-period comparison into a **continuous monitoring workflow**.
It reads the full parquet corpus, slices it into **yearly windows**,
and tracks spectral drift metrics over time producing a time-series of:

- **Wasserstein-1** distance of each year's spectrum vs. a fixed reference (1999-2007)
- **Spectral gap** per year
- **Mean eigenvalue** per year

> **Prerequisite**: `data/data/cve_embs/cve_corpus.parquet` must exist (see `cve.ipynb`).

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.stats import wasserstein_distance, gaussian_kde
from sklearn.neighbors import kneighbors_graph
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = DATA / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

K = 10
N_EIGS = 100
N_SAMPLE_PER_YEAR = 2_000
REFERENCE_YEARS = range(1999, 2008)  # baseline window
SEED = 42

## 1. Load Full Corpus

In [ ]:
df = pd.read_parquet(DATA / 'data' / 'cve_embs' / 'cve_corpus.parquet')

def _coerce(series):
    arr = series.to_numpy()
    if arr.dtype == object:
        return np.vstack(arr.tolist()).astype(np.float32)
    return arr.astype(np.float32)

print(f'Total CVEs: {len(df):,}')
print(f'Year range: {df.year.min()} – {df.year.max()}')

## 2. Build Reference Spectrum (1999-2007)

In [ ]:
def build_laplacian(X, k=K):
    A = kneighbors_graph(X, n_neighbors=k, mode='connectivity',
                         include_self=False, n_jobs=-1)
    A = (A + A.T).astype(np.float32); A.data[:] = 1.0
    d = np.asarray(A.sum(1)).flatten()
    d_inv_sqrt = np.where(d > 0, 1/np.sqrt(d), 0)
    D = sp.diags(d_inv_sqrt)
    return (sp.eye(len(X)) - D @ A @ D).tocsr()

def spectrum(X, k=K, n_eigs=N_EIGS):
    L = build_laplacian(X, k)
    ne = min(n_eigs, L.shape[0]-2)
    eigs, _ = spla.eigsh(L, k=ne, which='SM', tol=1e-4, maxiter=3000)
    return np.sort(np.real(eigs))

rng = np.random.default_rng(SEED)
ref_embs = _coerce(df[df.year.isin(REFERENCE_YEARS)].embedding)
ref_idx  = rng.choice(len(ref_embs), size=min(N_SAMPLE_PER_YEAR*3, len(ref_embs)), replace=False)
eigs_ref = spectrum(ref_embs[ref_idx])
print(f'Reference spectrum computed on {len(ref_idx):,} samples.')

## 3. Yearly Sweep

In [ ]:
years = sorted(df.year.unique())
records = []

for yr in years:
    embs_yr = _coerce(df[df.year == yr].embedding)
    if len(embs_yr) < K + 2:
        continue
    idx = rng.choice(len(embs_yr), size=min(N_SAMPLE_PER_YEAR, len(embs_yr)), replace=False)
    eigs_yr = spectrum(embs_yr[idx])
    gap = eigs_yr[eigs_yr > 1e-6][0] if np.any(eigs_yr > 1e-6) else 0.0
    records.append({
        'year': yr,
        'w1_vs_ref': wasserstein_distance(eigs_ref, eigs_yr),
        'spectral_gap': gap,
        'mean_eig': float(np.mean(eigs_yr)),
        'n_cves': len(embs_yr),
    })
    print(f'{yr}: W₁={records[-1]["w1_vs_ref"]:.5f}  gap={gap:.5f}  n={len(embs_yr):,}')

drift_df = pd.DataFrame(records)
drift_df.to_csv(RESULTS / 'yearly_spectral_drift.csv', index=False)
drift_df.head(10)

## 4. Time-Series Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(drift_df.year, drift_df.w1_vs_ref, color='tomato', lw=2, marker='o', ms=4)
axes[0].axvline(2015, color='gray', ls='--', lw=1, label='2015 split')
axes[0].set_ylabel('Wasserstein-1 vs. ref')
axes[0].set_title('Spectral Drift Monitoring — CVE Corpus (yearly)')
axes[0].legend(fontsize=8)

axes[1].plot(drift_df.year, drift_df.spectral_gap, color='steelblue', lw=2, marker='s', ms=4)
axes[1].axvline(2015, color='gray', ls='--', lw=1)
axes[1].set_ylabel('Spectral gap λ₂')

axes[2].bar(drift_df.year, drift_df.n_cves, color='slategray', alpha=0.6)
axes[2].set_ylabel('# CVEs published')
axes[2].set_xlabel('Year')
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(2))

plt.tight_layout()
plt.savefig(RESULTS / 'yearly_drift_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/results/yearly_drift_timeseries.png')